# Buried vs exposed significance sweep (¹⁵N error) — PANAV test set, 10-seed ensemble (ESM3)

Overall, per SS, per residue and residue × SS: protein-cluster bootstrap + Mann-Whitney, Bonferroni corrected. `delta = median_buried - median_exposed`.

In [ ]:
import os
os.makedirs("Figs", exist_ok=True)  # figures are written to ./Figs

import json
import numpy as np
import pandas as pd
from scipy.stats import mannwhitneyu

SS_ORDER       = ['helix', 'sheet', 'coil']
EXPOSURE_ORDER = ['buried', 'exposed']

PRED = '../../../data/analysis/Fig4/results/predictions_N_seed_ensemble_10seed_esm3.json'
GT   = '../../../data/rBMRB/cluster_30p/PANAV/test.json'
PRR  = '../../../data/analysis/Fig4/sasa_results/per_residue_records.json'

In [ ]:
# ---------- load ----------
with open(PRED) as f: pred_data = json.load(f)
with open(GT)   as f: gt_data   = json.load(f)
with open(PRR)  as f: prr_data  = json.load(f)

pred_dict, mask_dict = pred_data['mean'], pred_data['mask']   # 10-seed ensemble mean, not per-seed 'pred'
dictionary = {v: k for k, v in gt_data['entry_ID'].items()}

# ---------- build df ----------
records = []
for entry in gt_data['entry_ID'].values():
    key = dictionary.get(entry)
    if key is None or entry not in pred_dict:
        continue
    prr_entry = prr_data.get(entry)
    if prr_entry is None:
        continue

    gt_seq, gt_shifts = gt_data['sequence'][key], gt_data['N'][key]
    pred_arr_full  = np.array(pred_dict[entry])
    pred_mask_full = np.array(mask_dict[entry], dtype=bool)
    n = len(gt_seq)
    # predictions are batch-padded -- truncate to n rather than requiring an exact length match
    if len(pred_arr_full) < n or len(pred_mask_full) < n:
        continue
    pred_arr = pred_arr_full[:n]
    pred_mask = pred_mask_full[:n]

    prr_lookup = {r['resnum']: r for r in prr_entry}

    # Proline / None exclusion + the prediction's own mask (tag/flag/outlier exclusion already folded in)
    gt_mask_1 = np.array([res != 'P' for res in gt_seq])
    none_mask = np.array([g is not None for g in gt_shifts])
    mask      = gt_mask_1 & none_mask & pred_mask
    if mask.sum() == 0:
        continue

    full_indices = np.where(mask)[0]
    pred_f = pred_arr[mask]
    gt_f   = np.array(gt_shifts)[mask].astype(float)
    abs_error = np.abs(pred_f - gt_f)   # no analytical centering (PANAV referencing already consistent)

    for i, full_idx in enumerate(full_indices):
        resnum = int(full_idx) + 1
        rec    = prr_lookup.get(resnum)
        if rec is None or rec.get('ss_class') is None or rec.get('exposure') is None:
            continue
        records.append({
            'entry':        entry,
            'residue_type': gt_seq[full_idx],
            'ss_class':     rec['ss_class'],
            'exposure':     rec['exposure'],
            'abs_error':    abs_error[i],
        })

df = pd.DataFrame(records)
EXPOSURE_ORDER = [e for e in EXPOSURE_ORDER if e in set(df['exposure'])]
print(f'Total residues: {len(df)} | proteins: {df.entry.nunique()}')
print('Exposure split:', df['exposure'].value_counts().to_dict())

In [ ]:
# ---------- protein-cluster bootstrap ----------
def cluster_boot_delta(sub, gA, gB, col="exposure", B=2000, seed=0):
    """Bootstrap median(gB)-median(gA) by RESAMPLING PROTEINS (entry) with replacement."""
    ent = sub["entry"].to_numpy()
    val = sub["abs_error"].to_numpy()
    grp = sub[col].to_numpy()
    obs = np.median(val[grp == gB]) - np.median(val[grp == gA])
    uniq = np.unique(ent)
    pidx = {p: np.where(ent == p)[0] for p in uniq}
    rng = np.random.default_rng(seed)
    d = []
    for _ in range(B):
        samp = rng.choice(uniq, uniq.size, replace=True)
        idx = np.concatenate([pidx[p] for p in samp])
        a = val[idx][grp[idx] == gA]
        b = val[idx][grp[idx] == gB]
        if len(a) < 5 or len(b) < 5:
            continue
        d.append(np.median(b) - np.median(a))
    d = np.array(d)
    lo, hi = np.percentile(d, [2.5, 97.5])
    p = 2 * min((d <= 0).mean(), (d >= 0).mean())
    return obs, d.mean(), lo, hi, p, len(d)

In [ ]:
# ================= sweep config =================
MIN_N   = 30      # min sample per side (buried / exposed)
B_BOOT  = 100000    # bootstrap iters per contrast -- with strict Bonferroni, a hit needs boot_p well
                   # below 1/B_BOOT * n_tests to be distinguishable from the bootstrap's own resolution;
                   # bump this if a borderline p sits right at the 1/B_BOOT floor (see note below)
USE_BOOT = True   # False -> MWU only, much faster

def bonferroni(p, n_tests):
    """Family-wise error control: p_adj = min(p * n_tests, 1). Much stricter than BH-FDR
    (which controls the expected FALSE DISCOVERY RATE among rejected hypotheses, not the
    probability of ANY false positive across the whole family) -- every contrast is penalized
    by the full number of tests regardless of how many other contrasts look significant."""
    p = np.asarray(p, float)
    return np.minimum(p * n_tests, 1.0)

def test_be(sub, level, label):
    s = sub[sub.exposure.isin(["buried", "exposed"])]
    b = s[s.exposure == "buried"].abs_error.values
    e = s[s.exposure == "exposed"].abs_error.values
    if len(b) < MIN_N or len(e) < MIN_N:
        return None
    row = dict(level=level, group=label,
               n_buried=len(b), n_exposed=len(e), n_prot=s.entry.nunique(),
               med_buried=float(np.median(b)), med_exposed=float(np.median(e)),
               delta=float(np.median(b) - np.median(e)),
               p_mwu=float(mannwhitneyu(b, e, alternative="two-sided").pvalue))
    if USE_BOOT:
        obs, mu, lo, hi, pb, nvalid = cluster_boot_delta(s, "exposed", "buried", B=B_BOOT, seed=0)
        row.update(boot_delta=float(obs), ci_lo=float(lo), ci_hi=float(hi),
                   boot_p=float(pb), ci_excl0=bool(lo > 0 or hi < 0))
    return row

In [ ]:
# ================= sweep over all strata =================
rows = []
rows.append(test_be(df, "overall", "all"))                                   # 1) overall
for ss in SS_ORDER:                                                          # 2) per SS
    rows.append(test_be(df[df.ss_class == ss], "SS", ss))
for aa in sorted(df.residue_type.unique()):                                  # 3) per residue
    rows.append(test_be(df[df.residue_type == aa], "residue", aa))
for aa in sorted(df.residue_type.unique()):                                  # 4) residue x SS
    for ss in SS_ORDER:
        rows.append(test_be(df[(df.residue_type == aa) & (df.ss_class == ss)], "residue x SS", f"{aa}-{ss}"))

T = pd.DataFrame([r for r in rows if r is not None]).reset_index(drop=True)
n_tests = len(T)
T["p_bonf_mwu"] = bonferroni(T["p_mwu"].values, n_tests)
if USE_BOOT:
    T["p_bonf_boot"] = bonferroni(T["boot_p"].values, n_tests)
T["abs_delta"] = T["delta"].abs()
sort_p = "p_bonf_boot" if USE_BOOT else "p_bonf_mwu"
T = T.sort_values([sort_p, "abs_delta"], ascending=[True, False]).reset_index(drop=True)
print(f"{n_tests} contrasts tested  ->  Bonferroni threshold per test: p < {0.01/n_tests:.2e}")

In [ ]:
# ================= output =================
show_cols = ["level", "group", "n_buried", "n_exposed", "n_prot",
             "med_buried", "med_exposed", "delta", "p_mwu", "p_bonf_mwu"]
if USE_BOOT:
    show_cols += ["ci_lo", "ci_hi", "boot_p", "p_bonf_boot", "ci_excl0"]

pd.set_option("display.width", 240, "display.max_columns", 30)
print("--- Top 20 by significance ---")
print(T.head(20)[show_cols].round(4).to_string(index=False))

sig = T[T[sort_p] < 0.01]
print(f"\n--- significant after Bonferroni ({sort_p}<0.01): {len(sig)}/{len(T)} ---")
print(sig[show_cols].round(4).to_string(index=False))

print("\n--- most significant per level ---")
print(T.loc[T.groupby('level')[sort_p].idxmin()].sort_values(sort_p)[show_cols].round(4).to_string(index=False))

# note: boot_p is an empirical p-value from B_BOOT resamples, floored at 1/B_BOOT (2000 -> 0.0005);
# a row showing p_bonf_boot==0.0 exactly means "not a single resample crossed zero", i.e. p < 1/B_BOOT,
# not a mathematically exact zero -- increase B_BOOT for finer resolution near the Bonferroni threshold

T.to_csv("../../../data/analysis/Fig4/be_sweep_bonferroni_results_panav10seedens_esm3.csv", index=False)
print("\nsaved be_sweep_bonferroni_results_panav10seedens_esm3.csv")

## Forest plot: Bonferroni-significant strata

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


def set_style(base=13, family="Arial"):
    plt.rcParams.update({
        "font.family": family, "font.size": base,
        "axes.titlesize": base, "axes.labelsize": base,
        "xtick.labelsize": base - 2, "ytick.labelsize": base - 2,
        "legend.fontsize": base - 2, "svg.fonttype": "none",
    })


set_style(16)

T = pd.read_csv("../../../data/analysis/Fig4/be_sweep_bonferroni_results_panav10seedens_esm3.csv")

SIG_P = 0.01

sel = (T.p_bonf_boot < SIG_P) & (T.p_bonf_mwu < SIG_P) & (T.ci_excl0)
sub = T[sel].copy().sort_values("delta").reset_index(drop=True)
print(f"Bonferroni-significant strata: {len(sub)} / {len(T)}")

C_BUR, C_EXP, C_NS = "#4FA8A0", "#61AFEF", "#BFBFBF"            # buried worse / exposed worse / n.s.


def color(r):
    if r.p_bonf_boot >= SIG_P:
        return C_NS
    return C_BUR if r.delta > 0 else C_EXP


cols = [color(r) for r in sub.itertuples()]

y = np.arange(len(sub))
fig, ax = plt.subplots(figsize=(9, 0.46 * len(sub) + 1.4))
ax.errorbar(sub.delta, y, xerr=[sub.delta - sub.ci_lo, sub.ci_hi - sub.delta],
            fmt="none", ecolor="#9A9A9A", elinewidth=1.4, capsize=3, zorder=2)
ax.scatter(sub.delta, y, c=cols, s=90, edgecolor="k", linewidths=0.6, zorder=3)
ax.axvline(0, color="k", lw=1, ls="--", zorder=1)

ax.set_yticks(y)
ax.set_yticklabels(list(sub.group))
# for tick, r in zip(ax.get_yticklabels(), sub.itertuples()):
#     if r.p_bonf_boot < SIG_P:
#         tick.set_fontweight("bold")

xr = sub.ci_hi.max()
for yi, r in zip(y, sub.itertuples()):
    star = "*" if r.p_bonf_boot < SIG_P else ""
    ax.text(xr * 1.02 + 0.02, yi, f"p={r.p_bonf_boot:.3g}{star}  n={r.n_buried}/{r.n_exposed}",
            va="center", ha="left", fontsize=12, font="Arial",
            color="k" if r.p_bonf_boot < SIG_P else "#888")

# ax.text(0.015, 0.985, "← exposed worse", transform=ax.transAxes,
#         ha="left", va="top", fontsize=10, color=C_EXP, fontweight="bold")
# x in data units (ppm), y in axes fraction
ax.annotate("buried worse →",
            xy=(0, 0.985), xycoords=ax.get_xaxis_transform(),   # anchor: x=0 (data), top (axes fraction)
            xytext=(8, 0), textcoords="offset points",          # shift 8 pt right, independent of data scale
            ha="left", va="top", fontsize=12, color=C_BUR, fontweight="bold")

ax.set_xlabel(r"$\Delta$ median |error|:  buried $-$ exposed  (ppm), 95% CI")
ax.set_xlim(sub.ci_lo.min() - 0.1, xr * 1.4)
ax.set_ylim(-0.7, len(sub) - 0.3)
ax.set_title("Burial effect on ESM3 (struture + sequence) $^{15}$N prediction error")
for s in ("top", "right"):
    ax.spines[s].set_visible(False)
plt.tight_layout()
fig.savefig("Figs/burial_forest_residue_x_ss_bonferroni_panav10seedens_esm3.svg", bbox_inches="tight")
fig.savefig("Figs/burial_forest_residue_x_ss_bonferroni_panav10seedens_esm3.png", dpi=300, bbox_inches="tight")
plt.show()
print(f"saved burial_forest_residue_x_ss_bonferroni_panav10seedens_esm3  (n_strata={len(sub)})")
print(sub[["group", "delta", "ci_lo", "ci_hi", "p_bonf_boot", "n_buried", "n_exposed"]].round(4).to_string(index=False))